# WS4 — GRPO Training (Path A: Unsloth + TRL)

FarmSimulation hackathon, Person B. Trains `Qwen2.5-0.5B-Instruct` on Task 1 via TRL `GRPOTrainer` with Unsloth's vLLM-backed fast inference and 4-bit + LoRA.

**Status:** scaffold — cells 1–4 only. Cells 5–13 (FarmEnvClient, parse_action, 5 reward functions, GRPOConfig, trainer.train, plots, Hub upload) are pending HANDOFF #1 (A's WS1 merge — adds `narrative_text` to `FarmObservation`). Reference: `IMPLEMENTATION_PLAN.md` §20.3 / §20.7.

**Runtime:** Colab T4 (free tier OK for cells 1–4 smoke test). Final 50-iter GRPO training will run on the dedicated L4 Space at H+18-19.

**Critical flags (do not change without re-reading §20.3):**
- `fast_inference=True` requires `load_in_4bit=True` (vLLM backend, ~10× generation speedup).
- `use_gradient_checkpointing="unsloth"` — **string**, not `True`. Unsloth's custom impl saves an extra ~30% VRAM.
- `gpu_memory_utilization=0.7` — drop to `0.5` only if you OOM during rollout.

**Pin deviation from §20.7:** the plan's `trl==0.11.4` pin predates DAPO loss (TRL ≥0.13) and two-sided clipping (TRL ≥0.15) which §20.5 requires, and triggers a `UnslothGKDTrainer` SyntaxError under current `unsloth_zoo`. Cell 1 below pins `trl==0.22.2` + `transformers==4.56.2` to match Unsloth's official `Qwen2.5_(3B)-GRPO.ipynb` (April 2026, github.com/unslothai/notebooks).

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"   # let vLLM share GPU memory with training engine

# Aligned with Unsloth's official GRPO Colab notebook (April 2026):
#   github.com/unslothai/notebooks → nb/Qwen2.5_(3B)-GRPO.ipynb
# DEVIATION FROM §20.7: trl 0.11.4 → 0.22.2 (the plan pin predates DAPO loss / two-sided
# clipping required by §20.5, and triggers a UnslothGKDTrainer SyntaxError under
# current unsloth_zoo). transformers pinned to 4.56.2 per Unsloth's compat matrix.
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
!pip install --no-deps "trl==0.22.2" "transformers==4.56.2"
!pip install -U huggingface_hub openenv-core wandb

In [ ]:
from huggingface_hub import login as hf_login
hf_login()

import wandb
wandb.login()

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    max_seq_length=1104,                # = max_prompt_length (1024) + max_completion_length (80)
    load_in_4bit=True,                  # 4-bit base, LoRA stays bf16
    fast_inference=True,                # vLLM backend → ~10× faster generation
    max_lora_rank=16,
    gpu_memory_utilization=0.7,         # leave 30% for activations + KV cache
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,                      # alpha = 2 × r is Unsloth's default rule
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",   # NOT True/False — the string "unsloth" enables their custom impl
    random_state=3407,
)